In [22]:
import pandas as pd # Import the pandas library for data manipulation
import numpy as np # Import the numpy library for numerical operations
import re # Import the regular expression module for text cleaning
import string # Import the string module (though not directly used in the provided snippets, often useful for text processing)
import warnings # Import the warnings module to manage warnings
warnings.filterwarnings('ignore') # Ignore warning messages for cleaner output

from sklearn.model_selection import train_test_split # Import train_test_split for splitting data into training and testing sets
from sklearn.feature_extraction.text import TfidfVectorizer # Import TfidfVectorizer for converting text into TF-IDF features
from sklearn.linear_model import LogisticRegression # Import LogisticRegression for building the classification model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix # Import metrics for evaluating model performance

In [23]:
from google.colab import files # Import the files module from google.colab to handle file uploads
uploaded = files.upload() # Prompt the user to upload files and store the uploaded content

Saving Fake.csv to Fake (1).csv
Saving True.csv to True (1).csv


In [24]:
import pandas as pd # Import the pandas library again for data manipulation

fake_df = pd.read_csv('Fake.csv') # Load the 'Fake.csv' file into a pandas DataFrame named fake_df
true_df = pd.read_csv('True.csv') # Load the 'True.csv' file into a pandas DataFrame named true_df

print(fake_df.shape) # Print the dimensions (rows, columns) of the fake_df DataFrame
print(true_df.shape) # Print the dimensions (rows, columns) of the true_df DataFrame

(23481, 4)
(21417, 4)


In [25]:
fake_df['label'] = 0 # Add a new column 'label' to fake_df and set all its values to 0 (representing fake news)
true_df['label'] = 1 # Add a new column 'label' to true_df and set all its values to 1 (representing true news)

data = pd.concat([fake_df, true_df], axis=0) # Concatenate (combine) fake_df and true_df vertically into a single DataFrame named 'data'
data = data.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle the combined DataFrame randomly and reset its index

print('Combined dataset shape:', data.shape) # Print the dimensions of the combined dataset
data.head() # Display the first 5 rows of the combined DataFrame

Combined dataset shape: (44898, 5)


,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [26]:
data.isnull().sum() # Calculate and display the count of missing values for each column in the 'data' DataFrame

,0
title,0
text,0
subject,0
date,0
label,0


In [27]:
data['title'] = data['title'].fillna('') # Fill any missing values in the 'title' column with an empty string
data['text'] = data['text'].fillna('') # Fill any missing values in the 'text' column with an empty string
data['content'] = data['title'] + ' ' + data['text'] # Create a new 'content' column by concatenating 'title' and 'text'

data[['title', 'text', 'content', 'label']].head() # Display the first 5 rows of the 'title', 'text', 'content', and 'label' columns

,title,text,content,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,Trump drops Steve Bannon from National Securit...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",OOPS: Trump Just Accidentally Confirmed He Le...,0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",Donald Trump heads for Scotland to reopen a go...,1


In [28]:
def clean_text(text): # Define a function named clean_text that takes a text string as input
    text = str(text).lower() # Convert the text to string type and then to lowercase
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs from the text
    text = re.sub(r'\<.*?\>', '', text) # Remove HTML tags from the text
    text = re.sub(r'[^a-zA-Z\s]', ' ', text) # Remove non-alphabetic characters and replace with spaces
    text = re.sub(r'\s+', ' ', text).strip() # Replace multiple spaces with a single space and remove leading/trailing spaces
    return text # Return the cleaned text

data['content'] = data['content'].apply(clean_text) # Apply the clean_text function to the 'content' column of the DataFrame
data['content'].head() # Display the first 5 rows of the cleaned 'content' column

,content
0,ben stein calls out th circuit court committed...
1,trump drops steve bannon from national securit...
2,puerto rico expects u s to lift jones act ship...
3,oops trump just accidentally confirmed he leak...
4,donald trump heads for scotland to reopen a go...


In [29]:
X = data['content'] # Assign the 'content' column of the 'data' DataFrame to variable X (features)
y = data['label'] # Assign the 'label' column of the 'data' DataFrame to variable y (target)

print(X.shape) # Print the shape (number of samples) of X
print(y.shape) # Print the shape (number of samples) of y

(44898,)
(44898,)


In [30]:
X_train, X_test, y_train, y_test = train_test_split( # Split the data into training and testing sets
    X, y, test_size=0.2, random_state=42, stratify=y # 20% for testing, fixed random state for reproducibility, stratified splitting to maintain class distribution
)

print('Training samples:', len(X_train)) # Print the number of samples in the training features
print('Testing samples:', len(X_test)) # Print the number of samples in the testing features

Training samples: 35918
Testing samples: 8980


In [31]:
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7) # Initialize TfidfVectorizer with English stop words and a max document frequency of 0.7

X_train_tfidf = vectorizer.fit_transform(X_train) # Fit the vectorizer on the training data and transform it into TF-IDF features
X_test_tfidf = vectorizer.transform(X_test) # Transform the testing data into TF-IDF features using the fitted vectorizer

print('X_train_tfidf shape:', X_train_tfidf.shape) # Print the shape of the TF-IDF transformed training data
print('X_test_tfidf shape:', X_test_tfidf.shape) # Print the shape of the TF-IDF transformed testing data

X_train_tfidf shape: (35918, 101920)
X_test_tfidf shape: (8980, 101920)


In [32]:
lr_model = LogisticRegression(max_iter=1000) # Initialize a Logistic Regression model with a maximum of 1000 iterations
lr_model.fit(X_train_tfidf, y_train) # Train the Logistic Regression model using the TF-IDF training features and training labels

print('Model training completed.') # Print a message indicating that model training is complete

Model training completed.


In [33]:
y_pred = lr_model.predict(X_test_tfidf) # Use the trained Logistic Regression model to make predictions on the TF-IDF test features

In [34]:
accuracy = accuracy_score(y_test, y_pred) # Calculate the accuracy score by comparing true labels (y_test) with predicted labels (y_pred)
print('Accuracy:', round(accuracy * 100, 2), '%') # Print the accuracy, formatted as a percentage

print('\nClassification Report:\n') # Print a header for the classification report
print(classification_report(y_test, y_pred)) # Print the classification report, which includes precision, recall, f1-score, and support

print('Confusion Matrix:\n') # Print a header for the confusion matrix
print(confusion_matrix(y_test, y_pred)) # Print the confusion matrix, showing true positives, true negatives, false positives, and false negatives

Accuracy: 98.55 %

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4696
           1       0.98      0.99      0.98      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

Confusion Matrix:

[[4617   79]
 [  51 4233]]


In [35]:
def predict_news(news_text): # Define a function to predict whether a given news text is fake or real
    cleaned = clean_text(news_text) # Clean the input news text using the previously defined clean_text function
    vectorized = vectorizer.transform([cleaned]) # Transform the cleaned text into TF-IDF features using the trained vectorizer
    prediction = lr_model.predict(vectorized)[0] # Get the predicted label (0 for fake, 1 for real) from the model
    probability = lr_model.predict_proba(vectorized)[0] # Get the probability scores for each class (fake and real)
    label = 'Real News' if prediction == 1 else 'Fake News' # Assign a human-readable label based on the prediction
    return label, probability # Return the label and the class probabilities

sample_news = 'Government announces a new national education reform policy today.' # Define a sample news text for prediction
label, probability = predict_news(sample_news) # Call the predict_news function with the sample news

print('News:', sample_news) # Print the sample news text
print('Prediction:', label) # Print the predicted label for the sample news
print('Class Probabilities [Fake, Real]:', probability) # Print the class probabilities for the sample news

News: Government announces a new national education reform policy today.
Prediction: Fake News
Class Probabilities [Fake, Real]: [0.6413414 0.3586586]


In [36]:
import pickle # Import the pickle module for serializing and deserializing Python objects

with open('logistic_regression_fake_news_model.pkl', 'wb') as f: # Open a file in binary write mode to save the model
    pickle.dump(lr_model, f) # Serialize and save the trained logistic regression model to the file

with open('tfidf_vectorizer.pkl', 'wb') as f: # Open another file in binary write mode to save the vectorizer
    pickle.dump(vectorizer, f) # Serialize and save the TF-IDF vectorizer to the file

print('Model and vectorizer saved successfully.') # Print a success message

Model and vectorizer saved successfully.
